In [ ]:

# 1. Imports, configuration, reproducibility, and path discovery

import os
import glob
import json
import random
import copy
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import label_binarize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Deterministic settings improve reproducibility.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# The notebook searches common repository locations, so it works both
# when files are beside the notebook and when they live in a data/ folder.
SEARCH_ROOTS = [
    Path("."),
    Path("./data"),
    Path("./outputs"),
    Path("./processed_data"),
    Path("./01_data_preprocessing"),
]
# When running inside the ChatGPT workspace, uploaded shared artifacts
# are available under /mnt/data. This line is harmless elsewhere.
if Path("/mnt/data").exists():
    SEARCH_ROOTS.insert(0, Path("/mnt/data"))

def find_file(filename):
    for root in SEARCH_ROOTS:
        if root.exists():
            hits = list(root.rglob(filename))
            if hits:
                return hits[0]
    return None

LOGMEL_PATH = find_file("logmel.npy")
META_PATH = find_file("segments_meta.csv")
CFG_PATH = find_file("prep_config.json")

if LOGMEL_PATH is None or META_PATH is None or CFG_PATH is None:
    raise FileNotFoundError(
        "Could not locate logmel.npy, segments_meta.csv, and/or prep_config.json. "
        "Put the shared preprocessing outputs beside this notebook or inside ./data/."
    )

print("logmel:", LOGMEL_PATH)
print("metadata:", META_PATH)
print("config:", CFG_PATH)


In [ ]:

# 2. Load and validate the shared preprocessing artifacts

with open(CFG_PATH, "r", encoding="utf-8") as f:
    CFG = json.load(f)

CLASSES = CFG.get("classes", ["Healthy", "Chronic", "Non-chronic"])
N_CLASSES = len(CLASSES)

meta = pd.read_csv(META_PATH)
X = np.load(LOGMEL_PATH, mmap_mode="r")

assert len(X) == len(meta), (
    f"Mismatch: logmel has {len(X)} samples but metadata has {len(meta)} rows."
)
assert X.ndim == 3, f"Expected (N, mel, frames), got {X.shape}"
assert X.shape[1] == int(CFG["n_mels"]), "Mel-bin dimension does not match prep_config.json."
assert X.shape[2] == int(CFG["n_frames"]), "Frame dimension does not match prep_config.json."
assert {"label", "patient", "split"}.issubset(meta.columns)

# Validate labels.
if meta["label"].dtype.kind not in "iu":
    meta["label"] = meta["label"].astype(int)

assert set(meta["label"].unique()).issubset(set(range(N_CLASSES)))

print("X shape:", X.shape, X.dtype)
print("Classes:", dict(enumerate(CLASSES)))
print("\nSplit counts:")
print(meta["split"].value_counts())
print("\nClass counts:")
print(meta["label"].value_counts().sort_index().rename(index=dict(enumerate(CLASSES))))

print("\nPatients per split:")
print(meta.groupby("split")["patient"].nunique())

# Explicit leakage check.
train_patients = set(meta.loc[meta.split == "train", "patient"])
val_patients = set(meta.loc[meta.split == "val", "patient"])
test_patients = set(meta.loc[meta.split == "test", "patient"])

assert not train_patients & val_patients
assert not train_patients & test_patients
assert not val_patients & test_patients
print("\n✓ Patient leakage check passed.")


In [ ]:

# 3. Visual sanity check

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for label_id, ax in enumerate(axes):
    idxs = np.flatnonzero(meta["label"].to_numpy() == label_id)
    if len(idxs) == 0:
        ax.axis("off")
        continue
    idx = int(idxs[0])
    ax.imshow(np.asarray(X[idx], dtype=np.float32), origin="lower", aspect="auto")
    ax.set_title(f"{CLASSES[label_id]} — sample {idx}")
    ax.set_xlabel("Time frames")
    ax.set_ylabel("Mel bins")
plt.tight_layout()
plt.show()


In [ ]:

# 4. Train-only normalization statistics

train_idx = np.flatnonzero(meta["split"].to_numpy() == "train")
val_idx = np.flatnonzero(meta["split"].to_numpy() == "val")
test_idx = np.flatnonzero(meta["split"].to_numpy() == "test")

# Compute statistics in float32 to avoid float16 accumulation overflow.
train_sample = np.asarray(X[train_idx], dtype=np.float32)

# Global train-only statistics.
TRAIN_MEAN = float(train_sample.mean(dtype=np.float64))
TRAIN_STD = float(train_sample.std(dtype=np.float64))
TRAIN_STD = max(TRAIN_STD, 1e-6)

print(f"Train mean: {TRAIN_MEAN:.4f}")
print(f"Train std : {TRAIN_STD:.4f}")

del train_sample

def normalize_logmel(arr):
    arr = np.asarray(arr, dtype=np.float32)
    arr = (arr - TRAIN_MEAN) / TRAIN_STD
    return np.clip(arr, -5.0, 5.0).astype(np.float32)

print("✓ Normalization configured from training data only.")


In [ ]:

# 5. Dataset

class SpectrogramDataset(Dataset):
    def __init__(self, X, labels, indices, train=False,
                 freq_masks=2, freq_width=8, time_masks=2, time_width=25):
        self.X = X
        self.labels = np.asarray(labels)
        self.indices = np.asarray(indices)
        self.train = train
        self.freq_masks = freq_masks
        self.freq_width = freq_width
        self.time_masks = time_masks
        self.time_width = time_width

    def _spec_augment(self, x):
        x = x.copy()
        n_mels, n_frames = x.shape

        for _ in range(self.freq_masks):
            width = np.random.randint(0, min(self.freq_width, n_mels) + 1)
            if width > 0:
                start = np.random.randint(0, n_mels - width + 1)
                x[start:start + width, :] = 0.0

        for _ in range(self.time_masks):
            width = np.random.randint(0, min(self.time_width, n_frames) + 1)
            if width > 0:
                start = np.random.randint(0, n_frames - width + 1)
                x[:, start:start + width] = 0.0

        return x

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        x = normalize_logmel(self.X[idx])

        if self.train:
            x = self._spec_augment(x)

        # [1, mel, time]
        x = torch.from_numpy(x).unsqueeze(0)
        y = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return x, y, idx


labels = meta["label"].to_numpy(dtype=np.int64)

BATCH_SIZE = 64
NUM_WORKERS = 0  # safest across Windows/Jupyter

train_ds = SpectrogramDataset(X, labels, train_idx, train=True)
val_ds = SpectrogramDataset(X, labels, val_idx, train=False)
test_ds = SpectrogramDataset(X, labels, test_idx, train=False)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)

xb, yb, ib = next(iter(train_loader))
print("Batch:", xb.shape, yb.shape)


In [ ]:

# 6. Model

class SpectrogramTransformer(nn.Module):
    def __init__(
        self,
        n_mels=64,
        n_frames=251,
        patch_size=(8, 8),
        embed_dim=96,
        depth=2,
        num_heads=4,
        mlp_ratio=2.0,
        dropout=0.20,
        num_classes=3,
    ):
        super().__init__()

        ph, pw = patch_size
        self.patch_size = patch_size
        self.embed_dim = embed_dim

        # Padding only affects the time axis when n_frames is not divisible by pw.
        pad_h = (ph - n_mels % ph) % ph
        pad_w = (pw - n_frames % pw) % pw
        self.pad_h = pad_h
        self.pad_w = pad_w

        padded_h = n_mels + pad_h
        padded_w = n_frames + pad_w

        self.patch_embed = nn.Conv2d(
            1, embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
            bias=True
        )

        n_patches = (padded_h // ph) * (padded_w // pw)
        self.grid_h = padded_h // ph
        self.grid_w = padded_w // pw
        self.n_patches = n_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        self.last_patch_tokens = None

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        if self.pad_h or self.pad_w:
            x = F.pad(x, (0, self.pad_w, 0, self.pad_h))

        x = self.patch_embed(x)                 # [B, E, Gh, Gw]
        x = x.flatten(2).transpose(1, 2)       # [B, P, E]

        cls = self.cls_token.expand(x.size(0), -1, -1)
        tokens = torch.cat([cls, x], dim=1)
        tokens = tokens + self.pos_embed[:, :tokens.size(1)]
        tokens = self.pos_drop(tokens)

        # Retain gradients for Grad-CAM-style attribution.
        self.last_patch_tokens = tokens
        if tokens.requires_grad:
            tokens.retain_grad()

        encoded = self.encoder(tokens)
        cls_out = self.norm(encoded[:, 0])
        logits = self.head(cls_out)
        return logits


model = SpectrogramTransformer(
    n_mels=int(CFG["n_mels"]),
    n_frames=int(CFG["n_frames"]),
    num_classes=N_CLASSES,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {n_params:,}")

# Forward-pass sanity check before training.
with torch.no_grad():
    test_logits = model(xb[:2].to(DEVICE))
print("Forward-pass output:", test_logits.shape)
assert test_logits.shape == (2, N_CLASSES)
print("✓ Transformer forward pass passed.")


In [ ]:

# 7. Training utilities and class-weighted objective

# Inverse-frequency class weighting, normalized around mean weight 1.
class_counts = np.bincount(labels[train_idx], minlength=N_CLASSES).astype(np.float64)
class_weights = class_counts.sum() / (N_CLASSES * np.maximum(class_counts, 1))
class_weights = class_weights / class_weights.mean()

print("Training class counts:", dict(zip(CLASSES, class_counts.astype(int))))
print("Class weights:", dict(zip(CLASSES, np.round(class_weights, 3))))

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE),
    label_smoothing=0.05
)

LR = 3e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 25
PATIENCE = 6
GRAD_CLIP = 1.0

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

AMP_ENABLED = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)


def probabilities_from_logits(logits):
    return torch.softmax(logits, dim=1)


@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    all_probs, all_y, all_idx = [], [], []

    for xb, yb, ib in loader:
        xb = xb.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
            enabled=AMP_ENABLED
        ):
            logits = model(xb)

        probs = probabilities_from_logits(logits).cpu().numpy()
        all_probs.append(probs)
        all_y.append(yb.numpy())
        all_idx.append(ib.numpy())

    return (
        np.concatenate(all_probs),
        np.concatenate(all_y),
        np.concatenate(all_idx),
    )


def macro_f1(y_true, probs):
    pred = probs.argmax(axis=1)
    return precision_recall_fscore_support(
        y_true, pred, average="macro", zero_division=0
    )[2]


def run_epoch(model, loader, train=True):
    model.train(train)
    total_loss = 0.0
    total_n = 0

    for xb, yb, _ in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
            enabled=AMP_ENABLED
        ):
            logits = model(xb)
            loss = criterion(logits, yb)

        if train:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

        total_loss += float(loss.detach().cpu()) * len(yb)
        total_n += len(yb)

    return total_loss / max(total_n, 1)


In [ ]:

# 8. Train with validation macro-F1 early stopping

history = []
best_state = None
best_val_f1 = -np.inf
bad_epochs = 0

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, train=True)

    val_loss = run_epoch(model, val_loader, train=False)
    val_probs, val_y, _ = predict_loader(model, val_loader)

    val_pred = val_probs.argmax(1)
    val_acc = accuracy_score(val_y, val_pred)
    val_f1 = macro_f1(val_y, val_probs)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_macro_f1": val_f1,
        "lr": optimizer.param_groups[0]["lr"],
    })

    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {train_loss:.4f} | "
        f"val loss {val_loss:.4f} | "
        f"val acc {val_acc:.4f} | "
        f"val macro-F1 {val_f1:.4f}"
    )

    if val_f1 > best_val_f1 + 1e-4:
        best_val_f1 = val_f1
        best_state = copy.deepcopy(model.state_dict())
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= PATIENCE:
        print(f"Early stopping after {epoch} epochs.")
        break

elapsed = time.time() - start_time

assert best_state is not None
model.load_state_dict(best_state)

print(f"\nBest validation macro-F1: {best_val_f1:.4f}")
print(f"Training time: {elapsed/60:.2f} minutes")


In [ ]:

# 9. Training curves

hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="Train loss")
axes[0].plot(hist_df["epoch"], hist_df["val_loss"], label="Validation loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(hist_df["epoch"], hist_df["val_acc"], label="Validation accuracy")
axes[1].plot(hist_df["epoch"], hist_df["val_macro_f1"], label="Validation macro-F1")
axes[1].set_title("Validation metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:

# 10. Test evaluation

test_probs, test_y, test_indices = predict_loader(model, test_loader)
test_pred = test_probs.argmax(axis=1)

precision, recall, f1, _ = precision_recall_fscore_support(
    test_y, test_pred, labels=np.arange(N_CLASSES), zero_division=0
)

# One-vs-rest specificity for each class.
cm = confusion_matrix(test_y, test_pred, labels=np.arange(N_CLASSES))
specificities = []

for k in range(N_CLASSES):
    tp = cm[k, k]
    fn = cm[k, :].sum() - tp
    fp = cm[:, k].sum() - tp
    tn = cm.sum() - tp - fn - fp
    specificities.append(tn / max(tn + fp, 1))

try:
    multiclass_auroc = roc_auc_score(
        test_y,
        test_probs,
        multi_class="ovr",
        average="macro",
        labels=np.arange(N_CLASSES)
    )
except ValueError:
    multiclass_auroc = np.nan

test_metrics = {
    "model": "Attention Transformer",
    "accuracy": accuracy_score(test_y, test_pred),
    "macro_precision": precision.mean(),
    "macro_recall_sensitivity": recall.mean(),
    "macro_specificity": float(np.mean(specificities)),
    "macro_f1": f1.mean(),
    "auroc_ovr_macro": multiclass_auroc,
}

results_df = pd.DataFrame([test_metrics]).set_index("model")

print(results_df.T.to_string())
print("\nClassification report:")
print(
    classification_report(
        test_y,
        test_pred,
        labels=np.arange(N_CLASSES),
        target_names=CLASSES,
        zero_division=0
    )
)

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm)
ax.set_xticks(range(N_CLASSES), CLASSES, rotation=30, ha="right")
ax.set_yticks(range(N_CLASSES), CLASSES)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Attention Transformer — Test Confusion Matrix")

for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()


In [ ]:

# 11. Patient-level aggregation

test_meta = meta.iloc[test_indices].copy()
test_meta["true_label"] = test_y
for k, name in enumerate(CLASSES):
    test_meta[f"p_{name}"] = test_probs[:, k]

prob_cols = [f"p_{c}" for c in CLASSES]

patient_rows = []
for patient, g in test_meta.groupby("patient"):
    mean_probs = g[prob_cols].mean(axis=0).to_numpy()
    true_values = g["true_label"].unique()

    # Patient labels should be consistent under the preprocessing pipeline.
    true_label = int(pd.Series(true_values).mode().iloc[0])

    patient_rows.append({
        "patient": patient,
        "true_label": true_label,
        **{prob_cols[k]: mean_probs[k] for k in range(N_CLASSES)}
    })

patient_df = pd.DataFrame(patient_rows)
patient_probs = patient_df[prob_cols].to_numpy()
patient_true = patient_df["true_label"].to_numpy()
patient_pred = patient_probs.argmax(axis=1)

patient_f1 = precision_recall_fscore_support(
    patient_true, patient_pred, average="macro", zero_division=0
)[2]

print("Patients evaluated:", len(patient_df))
print("Patient-level accuracy:", round(accuracy_score(patient_true, patient_pred), 4))
print("Patient-level macro-F1:", round(patient_f1, 4))
print("\nPatient-level report:")
print(classification_report(
    patient_true, patient_pred,
    labels=np.arange(N_CLASSES),
    target_names=CLASSES,
    zero_division=0
))


In [ ]:

# 12. Monte Carlo Dropout

def enable_dropout_only(module):
    if isinstance(module, nn.Dropout):
        module.train()

@torch.no_grad()
def mc_dropout_predict(model, x, passes=20):
    model.eval()
    model.apply(enable_dropout_only)

    probs = []
    for _ in range(passes):
        logits = model(x)
        probs.append(torch.softmax(logits, dim=1))

    probs = torch.stack(probs, dim=0)  # [passes, batch, classes]
    mean_prob = probs.mean(dim=0)
    var_prob = probs.var(dim=0, unbiased=False)

    entropy = -(mean_prob * torch.log(mean_prob.clamp_min(1e-8))).sum(dim=1)
    pred = mean_prob.argmax(dim=1)
    confidence = mean_prob.max(dim=1).values
    winning_variance = var_prob.gather(1, pred.unsqueeze(1)).squeeze(1)

    model.eval()
    return mean_prob, entropy, confidence, winning_variance


# Demonstrate on the first test batch.
xb_demo, yb_demo, ib_demo = next(iter(test_loader))
xb_demo = xb_demo.to(DEVICE)

mc_prob, mc_entropy, mc_conf, mc_var = mc_dropout_predict(
    model, xb_demo[:16], passes=20
)

mc_demo = pd.DataFrame({
    "index": ib_demo[:16].numpy(),
    "true_label": yb_demo[:16].numpy(),
    "pred_label": mc_prob.argmax(1).cpu().numpy(),
    "confidence": mc_conf.cpu().numpy(),
    "entropy": mc_entropy.cpu().numpy(),
    "winning_prob_variance": mc_var.cpu().numpy(),
})

print(mc_demo.head(10).to_string(index=False))


In [ ]:

# 13. Transformer gradient-based attribution

def transformer_gradcam(model, x, target_class=None):
    model.eval()
    x = x.to(DEVICE).requires_grad_(True)

    model.zero_grad(set_to_none=True)
    logits = model(x)

    if target_class is None:
        target_class = int(logits.argmax(dim=1)[0].item())

    score = logits[:, target_class].sum()
    score.backward()

    tokens = model.last_patch_tokens
    grads = tokens.grad

    # Exclude CLS token.
    patch_tokens = tokens[:, 1:, :]
    patch_grads = grads[:, 1:, :]

    # Gradient × activation, then average over embedding dimension.
    cam = (patch_tokens * patch_grads).mean(dim=-1)
    cam = F.relu(cam)

    # [B, patches] -> [B, 1, grid_h, grid_w]
    cam = cam.reshape(
        cam.size(0), 1, model.grid_h, model.grid_w
    )

    cam = F.interpolate(
        cam,
        size=(int(CFG["n_mels"]), int(CFG["n_frames"])),
        mode="bilinear",
        align_corners=False
    )

    cam = cam[0, 0].detach().cpu().numpy()

    if cam.max() > cam.min():
        cam = (cam - cam.min()) / (cam.max() - cam.min())
    else:
        cam = np.zeros_like(cam)

    return cam, target_class, logits.detach().cpu().numpy()[0]


# Explain one held-out test sample.
sample_idx = int(test_indices[0])
sample_x = torch.from_numpy(
    normalize_logmel(X[sample_idx])
).unsqueeze(0).unsqueeze(0)

cam, explained_class, explained_logits = transformer_gradcam(
    model, sample_x
)

spec = normalize_logmel(X[sample_idx])

fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(spec, origin="lower", aspect="auto")
ax.imshow(cam, origin="lower", aspect="auto", alpha=0.45)
ax.set_title(
    f"Transformer time-frequency attribution — "
    f"predicted: {CLASSES[explained_class]}"
)
ax.set_xlabel("Time frames")
ax.set_ylabel("Mel bins")
plt.tight_layout()
plt.show()


In [ ]:

# 14. Error analysis

error_df = meta.iloc[test_indices].copy()
error_df["pred"] = test_pred
error_df["confidence"] = test_probs.max(axis=1)
error_df["correct"] = error_df["pred"] == error_df["label"]

print("Overall test accuracy:", round(error_df["correct"].mean(), 4))

if "device" in error_df.columns:
    print("\nAccuracy by device:")
    print(
        error_df.groupby("device")["correct"]
        .agg(["mean", "size"])
        .sort_values("mean")
        .round(4)
    )

if "cycle_label" in error_df.columns:
    print("\nAccuracy by cycle annotation:")
    print(
        error_df.groupby("cycle_label")["correct"]
        .agg(["mean", "size"])
        .sort_values("mean")
        .round(4)
    )

print("\nLowest-confidence test examples:")
display_cols = [
    c for c in ["recording", "patient", "device", "cycle_label", "label", "pred", "confidence"]
    if c in error_df.columns
]
print(
    error_df.sort_values("confidence")[display_cols]
    .head(15)
    .to_string(index=False)
)


In [ ]:

# 15A. External dataset configuration

EXTERNAL_MANIFEST = ""  # e.g. "./external/external_manifest.csv"

if not EXTERNAL_MANIFEST:
    print(
        "External manifest not supplied. Cross-dataset evaluation is skipped. "
        "Provide a CSV with path,label,dataset when the external dataset is available."
    )
else:
    external_manifest = pd.read_csv(EXTERNAL_MANIFEST)
    required_cols = {"path", "label", "dataset"}
    missing = required_cols - set(external_manifest.columns)
    if missing:
        raise ValueError(f"External manifest missing columns: {sorted(missing)}")

    external_manifest["label"] = external_manifest["label"].astype(int)
    assert set(external_manifest["label"].unique()).issubset({0, 1})
    print(external_manifest["dataset"].value_counts())
    print(external_manifest.head())


In [ ]:

# 15B. Optional audio adapter for external datasets
# This cell requires librosa. It is intentionally isolated from the ICBHI experiment.

if EXTERNAL_MANIFEST:
    import librosa
    from scipy.signal import butter, sosfiltfilt

    TARGET_SR = int(CFG["sr"])
    BAND = tuple(CFG["band"])
    SEG_SEC = float(CFG["seg_sec"])
    N_FFT = int(CFG["n_fft"])
    HOP = int(CFG["hop"])
    N_MELS = int(CFG["n_mels"])
    FMIN = int(CFG["fmin"])
    FMAX = min(int(CFG["fmax"]), TARGET_SR // 2)

    SOS = butter(5, BAND, btype="bandpass", fs=TARGET_SR, output="sos")

    def external_audio_to_logmel(path):
        y, sr = librosa.load(path, sr=TARGET_SR, mono=True)

        if len(y) == 0:
            raise ValueError(f"Empty audio: {path}")

        y = sosfiltfilt(SOS, y).astype(np.float32)

        target_len = int(SEG_SEC * TARGET_SR)
        if len(y) >= target_len:
            # Center crop for deterministic inference.
            start = (len(y) - target_len) // 2
            y = y[start:start + target_len]
        else:
            y = np.pad(y, (0, target_len - len(y)))

        mel = librosa.feature.melspectrogram(
            y=y,
            sr=TARGET_SR,
            n_fft=N_FFT,
            hop_length=HOP,
            n_mels=N_MELS,
            fmin=FMIN,
            fmax=FMAX,
            power=2.0
        )
        logmel = librosa.power_to_db(mel, ref=np.max)

        # Match the expected frame count exactly.
        n_frames = int(CFG["n_frames"])
        if logmel.shape[1] >= n_frames:
            logmel = logmel[:, :n_frames]
        else:
            logmel = np.pad(
                logmel,
                ((0, 0), (0, n_frames - logmel.shape[1])),
                mode="constant",
                constant_values=logmel.min()
            )

        return logmel.astype(np.float32)

    @torch.no_grad()
    def predict_external_audio(model, logmel):
        x = normalize_logmel(logmel)
        x = torch.from_numpy(x).unsqueeze(0).unsqueeze(0).to(DEVICE)
        probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
        return probs

    external_results = []

    for row in external_manifest.itertuples(index=False):
        path = str(row.path)
        if not os.path.exists(path):
            print("Missing:", path)
            continue

        try:
            lm = external_audio_to_logmel(path)
            probs = predict_external_audio(model, lm)

            # Collapse the ICBHI 3-class model into screening-level abnormality:
            # Healthy = 0, Chronic/Non-chronic = abnormal.
            abnormal_prob = float(probs[1:].sum())
            pred_binary = int(abnormal_prob >= 0.5)

            external_results.append({
                "path": path,
                "dataset": row.dataset,
                "true_label": int(row.label),
                "healthy_prob": float(probs[0]),
                "abnormal_prob": abnormal_prob,
                "pred_label": pred_binary,
            })
        except Exception as exc:
            print(f"Failed {path}: {exc}")

    external_results = pd.DataFrame(external_results)

    if len(external_results):
        for dataset_name, g in external_results.groupby("dataset"):
            y_ext = g["true_label"].to_numpy()
            p_ext = g["abnormal_prob"].to_numpy()
            pred_ext = g["pred_label"].to_numpy()

            print(f"\n===== {dataset_name} =====")
            print("N:", len(g))
            print("Accuracy:", round(accuracy_score(y_ext, pred_ext), 4))
            print(
                classification_report(
                    y_ext, pred_ext,
                    labels=[0, 1],
                    target_names=["Healthy", "Abnormal"],
                    zero_division=0
                )
            )

            if len(np.unique(y_ext)) == 2:
                print("AUROC:", round(roc_auc_score(y_ext, p_ext), 4))
    else:
        print("No valid external samples were evaluated.")


In [ ]:

# 16. Save model and results

OUTPUT_DIR = Path("./results_attention_transformer")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "classes": CLASSES,
    "config": CFG,
    "train_mean": TRAIN_MEAN,
    "train_std": TRAIN_STD,
    "model_kwargs": {
        "n_mels": int(CFG["n_mels"]),
        "n_frames": int(CFG["n_frames"]),
        "patch_size": (8, 8),
        "embed_dim": 96,
        "depth": 2,
        "num_heads": 4,
        "mlp_ratio": 2.0,
        "dropout": 0.20,
        "num_classes": N_CLASSES,
    }
}, OUTPUT_DIR / "attention_transformer_best.pt")

hist_df.to_csv(OUTPUT_DIR / "attention_training_history.csv", index=False)
results_df.to_csv(OUTPUT_DIR / "results_model4_attention.csv")

test_export = error_df.copy()
test_export.to_csv(OUTPUT_DIR / "attention_test_predictions.csv", index=False)

patient_df.to_csv(OUTPUT_DIR / "attention_patient_predictions.csv", index=False)

with open(OUTPUT_DIR / "attention_experiment_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "classes": CLASSES,
        "seed": SEED,
        "device": str(DEVICE),
        "train_mean": TRAIN_MEAN,
        "train_std": TRAIN_STD,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "epochs_max": EPOCHS,
        "patience": PATIENCE,
        "best_val_macro_f1": float(best_val_f1),
        "test_metrics": {k: (None if pd.isna(v) else float(v)) for k, v in test_metrics.items() if k != "model"},
    }, f, indent=2)

print("Saved to:", OUTPUT_DIR.resolve())
print(sorted(p.name for p in OUTPUT_DIR.iterdir()))


In [ ]:

# 17. Combined comparison table

candidate_files = {
    "Classical ML": "results_model1_classical.csv",
    "Custom CNN": "results_model2_cnn.csv",
    "CNN-BiLSTM": "results_model3_cnn_bilstm.csv",
}

comparison = [results_df.reset_index()]

for model_name, filename in candidate_files.items():
    path = find_file(filename)
    if path is not None:
        df = pd.read_csv(path)
        comparison.append(df)
        print("Loaded:", path)

comparison_df = pd.concat(comparison, ignore_index=True)

preferred_cols = [
    "model",
    "accuracy",
    "macro_precision",
    "macro_recall_sensitivity",
    "macro_specificity",
    "macro_f1",
    "auroc_ovr_macro",
]
comparison_df = comparison_df[
    [c for c in preferred_cols if c in comparison_df.columns]
]

display(comparison_df)

comparison_df.to_csv(
    OUTPUT_DIR / "all_model_comparison.csv",
    index=False
)
